In [ ]:
import os
import math
import json
import random
from glob import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------- paths --------
CLEAN_DIR = r"D:\Masters\Speech\Embeddings\clean"   # change this
PAIR_DIR  = r"D:\Masters\Speech\phonation_pairs"
MODEL_PATH = r"D:\Masters\Speech\reverse_phonation_best.pt"

# -------- forward diffusion settings --------
T_STEPS = 30
SCHEDULE = "cosine"

PHONATION_CFG = {
    "total_scale": 0.85,
    "noise_smooth_kernel": 11,
    "gain_smooth_kernel": 21,
    "stable_span_density_base": 0.03,
    "stable_span_density_scale": 0.10,
    "span_len_min": 4,
    "span_len_max": 12,
    "stability_power": 1.5,
    "max_gain_flutter": 0.16,      # multiplicative instability
    "max_jitter_scale": 0.22,      # additive smooth perturbation
    "max_breathy_weakening": 0.22, # local weakening toward local mean
    "max_coherence_loss": 0.30,    # degrade sustained coherence
}

In [ ]:
def moving_average_1d(x: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """
    Moving average along time axis for shape (T, D).
    """
    if kernel_size <= 1:
        return x.astype(np.float32)

    x = x.astype(np.float32)
    T, D = x.shape
    pad = kernel_size // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0)), mode="edge")

    out = np.zeros_like(x, dtype=np.float32)
    for t in range(T):
        out[t] = np.mean(x_pad[t:t + kernel_size], axis=0)
    return out.astype(np.float32)


def get_step_weights(T: int, schedule: str = "cosine") -> np.ndarray:
    """
    Returns normalized weights summing to 1.
    """
    if schedule == "uniform":
        w = np.ones(T, dtype=np.float32)
    elif schedule == "linear":
        w = np.arange(1, T + 1, dtype=np.float32)
    elif schedule == "cosine":
        t = np.linspace(0, 1, T, dtype=np.float32)
        w = 1.0 - np.cos((t + 1e-8) * np.pi / 2.0)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")

    w = np.maximum(w, 1e-8)
    w = w / np.sum(w)
    return w.astype(np.float32)


def cosine_similarity_mean(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    num = np.sum(a * b, axis=1)
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + eps
    return float(np.mean(num / den))


def evaluate_pair(clean: np.ndarray, corrupted: np.ndarray) -> dict:
    diff = corrupted - clean
    return {
        "mean_abs_error": float(np.mean(np.abs(diff))),
        "max_abs_error": float(np.max(np.abs(diff))),
        "mean_l2_per_frame": float(np.mean(np.linalg.norm(diff, axis=1))),
        "cosine": cosine_similarity_mean(clean, corrupted),
    }

In [ ]:
def phonation_forward_step_pronounced(
    x_prev: np.ndarray,
    step_weight: float,
    total_scale: float = 0.85,
    noise_smooth_kernel: int = 11,
    gain_smooth_kernel: int = 21,
    stable_span_density_base: float = 0.03,
    stable_span_density_scale: float = 0.10,
    span_len_min: int = 4,
    span_len_max: int = 12,
    stability_power: float = 1.5,
    max_gain_flutter: float = 0.16,
    max_jitter_scale: float = 0.22,
    max_breathy_weakening: float = 0.22,
    max_coherence_loss: float = 0.30,
    seed: int = 42
):
    """
    Pronounced phonation corruption in embedding space.

    Unlike articulation:
      - target stable / low-transition regions
      - add gain flutter
      - add smooth jitter
      - weaken local voiced spans
      - reduce coherence in sustained spans
    """
    rng = np.random.default_rng(seed)

    x_prev = x_prev.astype(np.float32)
    Tseq, D = x_prev.shape

  
    frame_diff = np.linalg.norm(x_prev[1:] - x_prev[:-1], axis=1)
    frame_diff = np.pad(frame_diff, (1, 0), mode="edge").astype(np.float32)

    stable_score = 1.0 / (frame_diff + 1e-4)
    stable_score = np.power(stable_score, stability_power)
    stable_prob = stable_score / np.sum(stable_score)

    num_spans = max(2, int((stable_span_density_base + stable_span_density_scale * step_weight) * Tseq))
    span_mask = np.zeros(Tseq, dtype=np.float32)

    for _ in range(num_spans):
        center = int(rng.choice(Tseq, p=stable_prob))
        span_len = int(rng.integers(span_len_min, span_len_max + 1))
        start = max(0, center - span_len // 2)
        end = min(Tseq, start + span_len)
        span_mask[start:end] = 1.0

    span_mask_2d = span_mask[:, None]

 
    raw_gain = rng.normal(0.0, 1.0, size=(Tseq, 1)).astype(np.float32)
    smooth_gain = moving_average_1d(raw_gain, kernel_size=gain_smooth_kernel)

    gain_flutter = max_gain_flutter * (0.35 + 0.65 * step_weight)
    gain_factor = 1.0 + total_scale * gain_flutter * smooth_gain
    gain_factor = np.clip(gain_factor, 0.70, 1.35)

    x_t = x_prev.copy()
    x_t = x_t + span_mask_2d * (x_prev * (gain_factor - 1.0))

    raw_noise = rng.normal(0.0, 1.0, size=(Tseq, D)).astype(np.float32)
    smooth_noise = moving_average_1d(raw_noise, kernel_size=noise_smooth_kernel)

    norms = np.linalg.norm(smooth_noise, axis=1, keepdims=True) + 1e-8
    direction = smooth_noise / norms

    frame_mag = np.linalg.norm(x_prev, axis=1, keepdims=True) + 1e-8
    jitter_scale = total_scale * max_jitter_scale * (0.30 + 0.70 * step_weight)

    jitter = span_mask_2d * (jitter_scale * 0.08 * frame_mag * direction)
    x_t = x_t + jitter

    breathy_alpha = max_breathy_weakening * (0.30 + 0.70 * step_weight)
    local_mean = moving_average_1d(x_prev, kernel_size=max(5, gain_smooth_kernel // 2))
    x_t = x_t + span_mask_2d * (breathy_alpha * (local_mean - x_t))


    coh_alpha = max_coherence_loss * (0.30 + 0.70 * step_weight)
    shift = int(rng.choice([-2, -1, 1, 2]))
    x_shift = np.roll(x_prev, shift=shift, axis=0)
    x_shift_smooth = moving_average_1d(x_shift, kernel_size=7)

    mix_ref = 0.5 * x_shift + 0.5 * x_shift_smooth
    x_t = x_t + span_mask_2d * (coh_alpha * (mix_ref - x_t))

    delta_t = x_t - x_prev

    meta = {
        "step_weight": float(step_weight),
        "num_spans": int(num_spans),
        "corrupted_fraction": float(np.mean(span_mask)),
        "delta_mean_l2": float(np.mean(np.linalg.norm(delta_t, axis=1))),
        "delta_max_l2": float(np.max(np.linalg.norm(delta_t, axis=1))),
        "gain_flutter": float(gain_flutter),
        "jitter_scale": float(jitter_scale),
        "breathy_alpha": float(breathy_alpha),
        "coh_alpha": float(coh_alpha),
    }

    return x_t.astype(np.float32), delta_t.astype(np.float32), meta

In [ ]:
def phonation_forward_diffusion_chain_pronounced(
    x_clean: np.ndarray,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **phonation_cfg
):
    """
    Builds x_0 -> x_1 -> ... -> x_T
    """
    x_clean = x_clean.astype(np.float32)
    x_states = [x_clean.copy()]
    deltas = [None]
    metas = [None]

    x = x_clean.copy()
    step_weights = get_step_weights(T, schedule=schedule)

    for t in range(1, T + 1):
        x, delta_t, meta_t = phonation_forward_step_pronounced(
            x_prev=x,
            step_weight=float(step_weights[t - 1]),
            seed=seed + t,
            **phonation_cfg
        )
        meta_t["t"] = t
        x_states.append(x)
        deltas.append(delta_t)
        metas.append(meta_t)

    return x_states, deltas, metas

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
print("Found:", len(sample_files))

x_clean = np.load(sample_files[0]).astype(np.float32)

x_states, deltas, metas = phonation_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PHONATION_CFG
)

x_T = x_states[-1]

print("clean -> x_T:", evaluate_pair(x_clean, x_T))
print("last meta:", metas[-1])

In [ ]:
def make_reverse_phonation_pairs_from_clean_folder(
    clean_dir: str,
    pair_dir: str,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **phonation_cfg
):
    os.makedirs(pair_dir, exist_ok=True)

    files = sorted(glob(os.path.join(clean_dir, "*.npy")))
    print("Number of clean files:", len(files))

    manifest = []

    for i, fpath in enumerate(files, 1):
        x_clean = np.load(fpath).astype(np.float32)

        x_states, deltas, metas = phonation_forward_diffusion_chain_pronounced(
            x_clean=x_clean,
            T=T,
            schedule=schedule,
            seed=seed + i,
            **phonation_cfg
        )

        base = os.path.splitext(os.path.basename(fpath))[0]

        for t in range(1, T + 1):
            save_path = os.path.join(pair_dir, f"{base}_t{t:02d}.npz")
            np.savez_compressed(
                save_path,
                x_t=x_states[t].astype(np.float32),
                x_prev=x_states[t - 1].astype(np.float32),
                t=np.array([t], dtype=np.int64),
                T=np.array([T], dtype=np.int64),
            )
            manifest.append(save_path)

        summary = evaluate_pair(x_clean, x_states[-1])
        print(
            f"[{i:04d}/{len(files):04d}] {base} | "
            f"clean->x_T cosine={summary['cosine']:.6f} | "
            f"mae={summary['mean_abs_error']:.6f}"
        )

    manifest_path = os.path.join(pair_dir, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("Saved manifest to:", manifest_path)
    print("Total training pairs:", len(manifest))

In [ ]:
make_reverse_phonation_pairs_from_clean_folder(
    clean_dir=CLEAN_DIR,
    pair_dir=PAIR_DIR,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PHONATION_CFG
)

In [ ]:
class ReversePhonationDataset(Dataset):
    def __init__(self, pair_dir: str):
        manifest_path = os.path.join(pair_dir, "manifest.json")
        with open(manifest_path, "r") as f:
            self.files = json.load(f)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = np.load(self.files[idx])
        x_t = item["x_t"].astype(np.float32)
        x_prev = item["x_prev"].astype(np.float32)
        t = int(item["t"][0])
        T = int(item["T"][0])

        return {
            "x_t": x_t,
            "x_prev": x_prev,
            "t": t,
            "T": T,
            "length": x_t.shape[0]
        }


def collate_reverse(batch):
    max_len = max(x["length"] for x in batch)
    D = batch[0]["x_t"].shape[1]

    x_t_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    x_prev_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    mask_batch = np.zeros((len(batch), max_len), dtype=np.float32)
    t_batch = np.zeros((len(batch),), dtype=np.int64)
    T_batch = np.zeros((len(batch),), dtype=np.int64)

    for i, item in enumerate(batch):
        L = item["length"]
        x_t_batch[i, :L] = item["x_t"]
        x_prev_batch[i, :L] = item["x_prev"]
        mask_batch[i, :L] = 1.0
        t_batch[i] = item["t"]
        T_batch[i] = item["T"]

    return {
        "x_t": torch.from_numpy(x_t_batch),
        "x_prev": torch.from_numpy(x_prev_batch),
        "mask": torch.from_numpy(mask_batch),
        "t": torch.from_numpy(t_batch),
        "T": torch.from_numpy(T_batch),
    }

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t_norm: torch.Tensor):
        device = t_norm.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=device).float() / max(half - 1, 1)
        )
        args = t_norm[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResidualBlock1D(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)

    def forward(self, x):
        res = x
        x = self.conv1(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = F.gelu(x)
        return x + res


class ReversePhonationUNetLite(nn.Module):
    def __init__(self, dim_in=1024, hidden=384, time_dim=128):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden)
        )

        self.in_proj = nn.Conv1d(dim_in, hidden, kernel_size=1)

        self.blocks = nn.ModuleList([
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
        ])

        self.out_proj = nn.Conv1d(hidden, dim_in, kernel_size=1)

    def forward(self, x_t, t, T):
        B, L, D = x_t.shape
        t_norm = t.float() / T.float().clamp(min=1.0)

        t_emb = self.time_embed(t_norm)
        t_emb = self.time_mlp(t_emb)

        x = x_t.transpose(1, 2)
        x = self.in_proj(x)
        x = x + t_emb[:, :, None]

        for blk in self.blocks:
            x = blk(x)

        delta_hat = self.out_proj(x).transpose(1, 2)
        x_prev_hat = x_t + delta_hat
        return x_prev_hat

In [ ]:
def masked_l1_l2_loss(pred, target, mask, l1_weight=1.0, l2_weight=1.0):
    mask = mask.unsqueeze(-1)

    l1 = torch.abs(pred - target) * mask
    l2 = (pred - target) ** 2 * mask

    denom = mask.sum().clamp(min=1.0) * pred.shape[-1]
    l1 = l1.sum() / denom
    l2 = l2.sum() / denom

    return l1_weight * l1 + l2_weight * l2, l1.detach(), l2.detach()


dataset = ReversePhonationDataset(PAIR_DIR)
n_total = len(dataset)
n_train = int(0.9 * n_total)
n_val = n_total - n_train

train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_set,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_reverse
)

val_loader = DataLoader(
    val_set,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_reverse
)

print("Train pairs:", len(train_set))
print("Val pairs:", len(val_set))

In [ ]:
model = ReversePhonationUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

best_val = float("inf")

for epoch in range(1, 21):
    # -------- train --------
    model.train()
    train_loss_sum = 0.0
    n_train_batches = 0

    for batch in train_loader:
        x_t = batch["x_t"].to(DEVICE)
        x_prev = batch["x_prev"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        t = batch["t"].to(DEVICE)
        T = batch["T"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(x_t, t, T)
        loss, _, _ = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        n_train_batches += 1

    train_loss = train_loss_sum / max(n_train_batches, 1)

    # -------- val --------
    model.eval()
    val_loss_sum = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            x_t = batch["x_t"].to(DEVICE)
            x_prev = batch["x_prev"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            t = batch["t"].to(DEVICE)
            T = batch["T"].to(DEVICE)

            pred = model(x_t, t, T)
            loss, _, _ = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)

            val_loss_sum += loss.item()
            n_val_batches += 1

    val_loss = val_loss_sum / max(n_val_batches, 1)

    print(f"Epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  Saved best model to", MODEL_PATH)

In [ ]:
best_model = ReversePhonationUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

print("Loaded:", MODEL_PATH)

In [ ]:
@torch.no_grad()
def reverse_phonation_diffusion(
    model: nn.Module,
    x_T: np.ndarray,
    T_steps: int,
    device: str = "cpu"
):
    x = torch.from_numpy(x_T.astype(np.float32)).unsqueeze(0).to(device)

    for t_cur in range(T_steps, 0, -1):
        t_tensor = torch.tensor([t_cur], dtype=torch.long, device=device)
        T_tensor = torch.tensor([T_steps], dtype=torch.long, device=device)
        x = model(x, t_tensor, T_tensor)

    return x.squeeze(0).cpu().numpy().astype(np.float32)

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
x_clean = np.load(sample_files[0]).astype(np.float32)

# forward phonation corruption
x_states, deltas, metas = phonation_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PHONATION_CFG
)
x_T = x_states[-1]

# reverse recovery
x_recovered = reverse_phonation_diffusion(
    model=best_model,
    x_T=x_T,
    T_steps=T_STEPS,
    device=DEVICE
)

print("clean -> x_T cosine:", cosine_similarity_mean(x_clean, x_T))
print("clean -> recovered cosine:", cosine_similarity_mean(x_clean, x_recovered))
print("identity MAE:", float(np.mean(np.abs(x_clean - x_T))))
print("recovered MAE:", float(np.mean(np.abs(x_clean - x_recovered))))

In [ ]:
test_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))[:20]
all_results = []

for i, fpath in enumerate(test_files, 1):
    x_clean = np.load(fpath).astype(np.float32)

    x_states, deltas, metas = phonation_forward_diffusion_chain_pronounced(
        x_clean=x_clean,
        T=T_STEPS,
        schedule=SCHEDULE,
        seed=SEED + i,
        **PHONATION_CFG
    )
    x_T = x_states[-1]

    x_recovered = reverse_phonation_diffusion(
        model=best_model,
        x_T=x_T,
        T_steps=T_STEPS,
        device=DEVICE
    )

    result = {
        "file": os.path.basename(fpath),
        "clean_to_xT_cos": cosine_similarity_mean(x_clean, x_T),
        "clean_to_rec_cos": cosine_similarity_mean(x_clean, x_recovered),
        "identity_mae": float(np.mean(np.abs(x_clean - x_T))),
        "recovered_mae": float(np.mean(np.abs(x_clean - x_recovered))),
    }
    all_results.append(result)

    print(
        f"[{i:02d}/{len(test_files):02d}] {result['file']} | "
        f"xT cos={result['clean_to_xT_cos']:.6f} | "
        f"rec cos={result['clean_to_rec_cos']:.6f} | "
        f"id mae={result['identity_mae']:.6f} | "
        f"rec mae={result['recovered_mae']:.6f}"
    )

print("\nAverages:")
print("clean -> x_T cosine:", np.mean([r["clean_to_xT_cos"] for r in all_results]))
print("clean -> recovered cosine:", np.mean([r["clean_to_rec_cos"] for r in all_results]))
print("identity MAE:", np.mean([r["identity_mae"] for r in all_results]))
print("recovered MAE:", np.mean([r["recovered_mae"] for r in all_results]))